# Phase 3: Statistical Analysis
### Student Dropout & Performance Prediction System
This notebook contains the **4 rigorous statistical tests** to analyze predictors of student dropout.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
# Load data from database
db_path = "../data/students.db"
if not os.path.exists(db_path):
    db_path = "data/students.db"

conn = sqlite3.connect(db_path)
df = pd.read_sql("SELECT * FROM students", conn)
conn.close()
print(f"Loaded dataset with {len(df)} records.")

### Test 1: Chi-Square Test of Independence
**Question:** Does scholarship status significantly affect whether a student drops out vs. graduates?
*(Excluding active Enrolled students to focus on final outcomes)*

In [ ]:
# Filter out Enrolled to test final outcomes (Dropout vs Graduate)
df_filtered = df[df['Target'].isin(['Dropout', 'Graduate'])]

# Create contingency table
contingency_table = pd.crosstab(df_filtered['Scholarship holder'], df_filtered['Target'])
print("Contingency Table (Scholarship status vs outcome):")
print(contingency_table)

# Run Chi-Square Test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
print(f"\nChi-Square Statistic: {chi2:.4f}")
print(f"P-value: {p:.4g}")

# Interpretation
alpha = 0.05
if p < alpha:
    print("\nResult: Statistically Significant (p < 0.05). Reject the null hypothesis.")
    print("Conclusion: Having a scholarship significantly affects the final dropout vs graduation rate.")
else:
    print("\nResult: Not Significant (p >= 0.05). Fail to reject the null hypothesis.")

### Test 2: Independent Samples T-Test
**Question:** Is there a significant difference in Admission grades between students who drop out vs. students who graduate?

In [ ]:
dropout_admission = df[df['Target'] == 'Dropout']['Admission grade']
graduate_admission = df[df['Target'] == 'Graduate']['Admission grade']

print(f"Mean Admission Grade (Dropouts): {dropout_admission.mean():.2f}")
print(f"Mean Admission Grade (Graduates): {graduate_admission.mean():.2f}")

# Run independent t-test
t_stat, p_value = stats.ttest_ind(dropout_admission, graduate_admission, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4g}")

# Interpretation
if p_value < alpha:
    print("\nResult: Statistically Significant (p < 0.05).")
    print("Conclusion: There is a highly significant difference in admission grades between dropouts and graduates.")
else:
    print("\nResult: Not Significant (p >= 0.05).")

### Test 3: Analysis of Variance (ANOVA)
**Question:** Do semester grades (e.g. 2nd semester approved grades) differ significantly across all 3 outcome groups (Dropout, Enrolled, Graduate)?

In [ ]:
dropout_g = df[df['Target'] == 'Dropout']['Curricular units 2nd sem (grade)']
enrolled_g = df[df['Target'] == 'Enrolled']['Curricular units 2nd sem (grade)']
graduate_g = df[df['Target'] == 'Graduate']['Curricular units 2nd sem (grade)']

print(f"Mean 2nd Sem Grade (Dropouts): {dropout_g.mean():.2f}")
print(f"Mean 2nd Sem Grade (Enrolled): {enrolled_g.mean():.2f}")
print(f"Mean 2nd Sem Grade (Graduates): {graduate_g.mean():.2f}")

# Run ANOVA
f_stat, p_val = stats.f_oneway(dropout_g, enrolled_g, graduate_g)
print(f"\nF-statistic: {f_stat:.4f}")
print(f"P-value: {p_val:.4g}")

# Interpretation
if p_val < alpha:
    print("\nResult: Statistically Significant (p < 0.05).")
    print("Conclusion: Semester grade averages vary significantly across Dropout, Enrolled, and Graduate outcomes.")
else:
    print("\nResult: Not Significant (p >= 0.05).")

### Test 4: Pearson Correlation
**Question:** Is there a linear relationship between macroeconomic conditions (GDP) and the numeric indicator of dropout?

In [ ]:
# Map target to binary value (Dropout = 1, Other = 0)
df['Is_Dropout'] = df['Target'].apply(lambda x: 1 if x == 'Dropout' else 0)

# Run Pearson Correlation
corr_coef, p_val_corr = stats.pearsonr(df['GDP'], df['Is_Dropout'])
print(f"Pearson Correlation Coefficient (GDP vs Dropout status): {corr_coef:.4f}")
print(f"P-value: {p_val_corr:.4g}")

# Interpretation
if p_val_corr < alpha:
    print("\nResult: Statistically Significant (p < 0.05).")
    print(f"Conclusion: GDP growth show a significant but very weak linear correlation (r = {corr_coef:.4f}) with student dropouts.")
else:
    print("\nResult: Not Significant (p >= 0.05).")